In [1]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import RobustScaler
import joblib

# ✅ Updated paths (YOUR FINAL CORRECT PATHS)
INPUT_CSV = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\eye_clean.csv"

X_OUTPUT = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\X_train.npy"
Y_OUTPUT = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\y_train.npy"
GROUPS_OUTPUT = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\groups.npy"

SCALER_PATH = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed\robust_scaler.pkl"

In [2]:
print("🚀 Loading data in chunks...")

chunks = pd.read_csv(INPUT_CSV, low_memory=False, chunksize=500000, decimal=",")
df_list = []

for i, chunk in enumerate(chunks):

    feature_cols = [c for c in chunk.columns if c not in ['Participant name', 'Recording name', 'Recording timestamp']]

    # Safe numeric conversion
    for col in feature_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors='coerce')

    # ✅ FIXED fill (no method= error)
    chunk[feature_cols] = chunk.groupby('Participant name')[feature_cols].ffill(limit=3)
    chunk[feature_cols] = chunk.groupby('Participant name')[feature_cols].bfill(limit=3)

    # Downcast
    fcols = chunk.select_dtypes('float').columns
    chunk[fcols] = chunk[fcols].astype('float32')

    df_list.append(chunk)
    print(f"✅ Chunk {i+1} processed")

df = pd.concat(df_list, axis=0)
del df_list
gc.collect()

print(f"✅ Loaded shape: {df.shape}")

🚀 Loading data in chunks...
✅ Chunk 1 processed
✅ Chunk 2 processed
✅ Chunk 3 processed
✅ Chunk 4 processed
✅ Chunk 5 processed
✅ Chunk 6 processed
✅ Chunk 7 processed
✅ Chunk 8 processed
✅ Chunk 9 processed
✅ Chunk 10 processed
✅ Loaded shape: (4844304, 39)


In [3]:
print("🧠 Creating features...")

gaze_diff = df.groupby('Participant name')[['Gaze point X', 'Gaze point Y']].diff()

df['velocity_px'] = np.sqrt(
    gaze_diff['Gaze point X']**2 + gaze_diff['Gaze point Y']**2
).fillna(0).astype('float32')

del gaze_diff

# Pupil feature
df['pupil_avg'] = (df['Pupil diameter left'] + df['Pupil diameter right']) / 2

group = df.groupby('Participant name')['pupil_avg']

df['pupil_z'] = (
    (df['pupil_avg'] - group.transform('mean')) /
    (group.transform('std') + 1e-6)
).astype('float32')

df.drop(columns=['pupil_avg'], inplace=True)

🧠 Creating features...


In [4]:
print("🏷️ Creating labels...")

conditions = [
    (df['pupil_z'] > 1.2) & (df['velocity_px'] > 15),
    (df['velocity_px'] > 10),
    (df['pupil_z'].abs() < 0.5),
]

choices = [3, 2, 1]

df['label'] = np.select(conditions, choices, default=0).astype('int8')

🏷️ Creating labels...


In [5]:
print("📏 Scaling...")

exclude = ['Participant name', 'Recording name', 'Recording timestamp', 'label']
feature_cols = [c for c in df.columns if c not in exclude]

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols].values.astype('float32'))

joblib.dump(scaler, SCALER_PATH)

📏 Scaling...


c:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
c:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


['D:\\ML_Project\\emotion-drift-project\\data\\eyet4empathy\\processed\\robust_scaler.pkl']

In [6]:
print("📦 Creating sequences...")

X, y, groups = [], [], []

SEQ_LEN = 100
STEP = 50

for p, group_df in df.groupby(['Participant name', 'Recording name']):

    feat_vals = group_df[feature_cols].values
    lab_vals = group_df['label'].values

    if len(feat_vals) < SEQ_LEN:
        continue

    for i in range(0, len(feat_vals) - SEQ_LEN, STEP):
        X.append(feat_vals[i:i+SEQ_LEN])
        y.append(np.bincount(lab_vals[i:i+SEQ_LEN]).argmax())
        groups.append(p)

print("✅ Sequences created")

📦 Creating sequences...
✅ Sequences created


In [7]:
print("💾 Saving data...")

X_final = np.array(X, dtype='float32')
y_final = np.array(y, dtype='int8')
groups_final = np.array(groups)

np.save(X_OUTPUT, X_final)
np.save(Y_OUTPUT, y_final)
np.save(GROUPS_OUTPUT, groups_final)

print("✅ Saved successfully")
print("X shape:", X_final.shape)
print("y shape:", y_final.shape)

💾 Saving data...
✅ Saved successfully
X shape: (96342, 100, 38)
y shape: (96342,)
